# Imports

In [168]:
import os
import json
import yaml
from pathlib import Path
import re
import datetime
from collections import defaultdict


import boto3
from dotenv import load_dotenv
import pandas as pd


# Environmental Variables

In [105]:
# set S3_BUCKET environmental variable to arn:aws:s3:::kasbench-test-20260528-377288663341-us-east-1-an
os.environ["S3_BUCKET"] = ""


In [18]:
load_dotenv()
s3_bucket = os.environ.get("S3_BUCKET")
print(f"S3 Bucket: {s3_bucket}")



S3 Bucket: kasbench-test-20260528-377288663341-us-east-1-an


# Constants

In [156]:
run_id = "verification-15-percentile-large-a"
s3 = boto3.client("s3")

data_directory = "../data"
output_directory = "../output"



monitored_events_text = """NodeNotReady
NodeUnreachable
NodeHasMemoryPressure
NodeHasDiskPressure
NodeHasPIDPressure
Evicted
OOMKilling
SystemOOM
FailedMount
FailedAttachVolume
FailedCreatePodSandBox
FailedPull
ErrImagePull
ImagePullBackOff
NetworkNotReady
FailedScheduling
Unhealthy
BackOff
CrashLoopBackOff
FailedCreate
FailedKillPod
FailedSync
Rebooted"""

monitored_events = [event for event in monitored_events_text.split("\n") if event]


# Functions

In [ ]:
def download_file_from_s3(s3_bucket, s3_key, local_path):
    """Download a file from S3 to a local path."""
    s3.download_file(s3_bucket, s3_key, local_path)
    print(f"Downloaded {s3_key} to {local_path}")


def get_s3_file_listing(s3_bucket, s3_prefix=''):
    """Get a listing of files in an S3 bucket."""
    file_list = []
    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=s3_bucket, Prefix=s3_prefix)
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                file_list.append(obj)
    return file_list

def get_s3_file(s3_bucket, s3_filename, local_filename, data_directory, force=False):
    """Download a file from S3 to a local path."""

    # rename the local file
    local_filename = f"{data_directory}/{local_filename}"

    # skip if file exists and not force
    if os.path.exists(local_filename) and not force: 
        return local_filename
    
    
    # create the local directory, if it doesn't exist 

    dir_path = str(Path(f"{data_directory}/{local_filename}").parent)
    os.makedirs(dir_path, exist_ok=True)

    # download the tile
    s3.download_file(s3_bucket, s3_filename, local_filename)
    return local_filename

def parse_trial(filename):
    pattern = r"^[^/]+/(trial\d{4})/"
    if match := re.match(pattern, filename):
        return match.group(1)
    return None

In [152]:
def get_trial_start_end_times(files, run_id, s3_bucket, data_directory, force=False, verbose=False):
    trials = {}
    for file in files:
        if trial_id := parse_trial((file["Key"])):
            s3_filename = f"{run_id}/{trial_id}/run_details.json"
            local_filename = get_s3_file(s3_bucket, s3_filename, s3_filename, data_directory, force=force)
            with open(local_filename, "r") as f:
                trial = json.load(f)
                start_time = trial["status"]["startTime"] # string in format '2026-09-18T21:18:54.489000+00:00'
                end_time = trial["status"]["endTime"] # string in format '2026-09-18T21:18:54.489000+00:00'
                # convert start_time and end_time to timezone-aware dateteime objects
                start_time = datetime.datetime.fromisoformat(start_time.replace('Z', '+00:00'))
                end_time = datetime.datetime.fromisoformat(end_time.replace('Z', '+00:00'))
                trials[trial_id] = {"start_time": start_time, "end_time": end_time}

    if verbose:
        for trial_id, times in trials.items():
            print(f"{trial_id}: {times['start_time']} -> {times['end_time']}")

    return trials


def get_run_details(s3_bucket, run_id, trial_id, verbose=False):
    """Get the run details from S3"""
    key = f"{run_id}/{trial_id}/run_details.json"
    obj = s3.get_object(Bucket=s3_bucket, Key=key)
    return json.loads(obj['Body'].read())


def get_autoscaler(run_details):
    """ Get the autoscaler from the run details"""
    autoscaler = run_details["initialization"]["autoscaler"]
    return autoscaler    
                

# Code

## Get the listing of files in the S3 bucket for the specified run

In [86]:
files = get_s3_file_listing(s3_bucket, run_id)


## Get the starting and ending time of each trial

In [89]:
trial_times = get_trial_start_end_times(files, run_id, s3_bucket, data_directory, force=False, verbose=False)        
trial_times

{'trial0001': {'start_time': datetime.datetime(2026, 9, 18, 20, 48, 48, 765649, tzinfo=datetime.timezone.utc),
  'end_time': datetime.datetime(2026, 9, 18, 21, 18, 54, 489000, tzinfo=datetime.timezone.utc)},
 'trial0002': {'start_time': datetime.datetime(2026, 9, 18, 21, 40, 44, 31588, tzinfo=datetime.timezone.utc),
  'end_time': datetime.datetime(2026, 9, 18, 22, 10, 49, 394000, tzinfo=datetime.timezone.utc)},
 'trial0003': {'start_time': datetime.datetime(2026, 9, 18, 22, 33, 0, 262451, tzinfo=datetime.timezone.utc),
  'end_time': datetime.datetime(2026, 9, 18, 23, 3, 5, 59000, tzinfo=datetime.timezone.utc)},
 'trial0004': {'start_time': datetime.datetime(2026, 9, 18, 23, 24, 46, 327266, tzinfo=datetime.timezone.utc),
  'end_time': datetime.datetime(2026, 9, 18, 23, 54, 50, 591000, tzinfo=datetime.timezone.utc)},
 'trial0005': {'start_time': datetime.datetime(2026, 9, 19, 0, 15, 53, 829765, tzinfo=datetime.timezone.utc),
  'end_time': datetime.datetime(2026, 9, 19, 0, 45, 59, 883000,

## Analyze event logs (warnings.yaml)

In [158]:
results = []
reasons = set()
exclude_probes = False
counter = 0
autoscalers = {}
                
for file in files:
    if "post/events/warnings.yaml" in file["Key"]:
        filename = file["Key"]
        
        # parse the trial id from the filename
        trial_id = parse_trial(filename)

        # get the autoscaler
        autoscaler = autoscalers.get(trial_id)
        if autoscaler is None:
            run_details = get_run_details(s3_bucket, run_id, trial_id)
            autoscaler = get_autoscaler(run_details)
            autoscalers[trial_id] = autoscaler
            print(f"Trial: {trial_id} \tAutoscaler: {autoscaler}")
        
        # download the event file
        filename = get_s3_file(s3_bucket, filename, filename, data_directory)
        
        # load the event file
        with open(filename, "r") as f:
            events = yaml.load(f, Loader=yaml.FullLoader)
            
        for item in events["items"]:
            reason = item["reason"]
            type = item["type"]
            
            if  True:
                
                first_timestamp = item["firstTimestamp"] # string in format 2026-09-18T20:39:54Z
                last_timestamp = item["lastTimestamp"] # string in format 2026-09-18T20:39:54Z
                
                
                if first_timestamp is None or last_timestamp is None:
                    contineu

                # convert first_timestamp and last_timestamp to timezone aware datetimes
                first_timestamp = datetime.datetime.fromisoformat(first_timestamp.replace('Z', '+00:00'))
                last_timestamp = datetime.datetime.fromisoformat(last_timestamp.replace('Z', '+00:00'))
                
                # exclude events before and after the trial run
                if last_timestamp < trial_times[trial_id]["start_time"] or first_timestamp > trial_times[trial_id]["end_time"]:
                    continue

                reasons.add(reason)
            

                # gather relevant fields
                action = item.get("action")
                kind = item["involvedObject"]["kind"]
                name = item["involvedObject"]["name"]
                namespace = item["involvedObject"]["namespace"]
                message = item["message"]
                type = item["type"]
                count = item["count"]

                # exclude probe failures in the globeco namespace
                if exclude_probes and namespace == "globeco" and re.search("^(Readiness|Startup|Liveness) probe failed", message):
                    continue
                

                counter += 1
                print(f"*** {type}: {reason} ***")
                print(autoscaler)
                print(f"{first_timestamp} {last_timestamp}")
                print(f"{kind} {name} {namespace}")
                print(message)
                print(f"Count: {count}")
                print(f"Duration: {last_timestamp - first_timestamp}")
                # print(json.dumps(item, indent=2))
                print()

                results.append({
                    "action": action,
                    "type": type,
                    "reason": reason,
                    "count": count,
                    "first_timestamp": first_timestamp,
                    "last_timestamp": last_timestamp,
                    "trial_start_time": trial_times[trial_id]["start_time"],
                    "trial_end_time": trial_times[trial_id]["end_time"],
                    "duration": last_timestamp - first_timestamp,
                    "kind": kind,
                    "name": name,
                    "namespace": namespace,
                    "message": message,
                    "trial_id": trial_id,
                    "autoscaler": autoscaler,
                })
                
        # print(f"Trial: {trial_id} \tCounter: {counter}")    
        

print(reasons)    
print(results)
print(counter)

# write results array to a json file in the output directory
with open(f"{output_directory}/audit_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

Trial: trial0001 	Autoscaler: hpa
*** Warning: Unhealthy ***
hpa
2026-09-18 20:47:38+00:00 2026-09-18 20:49:38+00:00
Pod globeco-fix-engine-d95677f59-9smgj globeco
Startup probe failed: Get "http://10.244.2.19:8085/healthz": dial tcp 10.244.2.19:8085: connect: connection refused
Count: 25
Duration: 0:02:00

*** Warning: FailedGetResourceMetric ***
hpa
2026-09-18 20:45:56+00:00 2026-09-18 20:50:41+00:00
HorizontalPodAutoscaler globeco-fix-engine-hpa globeco
failed to get cpu utilization: unable to get metrics for resource cpu: no metrics returned from resource metrics API
Count: 9
Duration: 0:04:45

*** Warning: Unhealthy ***
hpa
2026-09-18 21:02:27+00:00 2026-09-18 21:02:27+00:00
Pod globeco-order-generation-service-54cd6688b5-ndtd4 globeco
Readiness probe failed: Get "http://10.244.1.27:8088/health/ready": dial tcp 10.244.1.27:8088: connect: connection refused
Count: 1
Duration: 0:00:00

*** Warning: Unhealthy ***
hpa
2026-09-18 20:50:52+00:00 2026-09-18 20:50:52+00:00
Pod globeco-ord

In [103]:
results

[]

In [163]:
count = 0
for result in results:
    if result["reason"] == "FailedGetResourceMetric" and result["name"] != "globeco-portfolio-service-hpa":
        print(result["name"], result["trial_id"], result["count"], result["autoscaler"])
        count += 1
print(count)

globeco-fix-engine-hpa trial0001 9 hpa
globeco-order-generation-service-hpa trial0004 1 keda
globeco-order-service-hpa trial0004 8 keda
globeco-portfolio-accounting-service-hpa trial0004 8 keda
globeco-security-service-hpa trial0004 1 keda
globeco-fix-engine-hpa trial0008 13 hpa
globeco-execution-service-hpa trial0013 1 keda
globeco-order-generation-service-hpa trial0013 8 keda
globeco-order-service-hpa trial0013 1 keda
globeco-portfolio-accounting-service-hpa trial0013 8 keda
globeco-fix-engine-hpa trial0014 9 hpa
globeco-allocation-service-hpa trial0015 1 keda
globeco-trade-service-hpa trial0015 1 keda
13


In [166]:
autoscalers = ["keda", "hpa"]
for autoscaler in autoscalers:
    count = 0
    for result in results:
        if result["autoscaler"] == autoscaler:
            print(result["name"], result["trial_id"], result["count"], result["autoscaler"])
            count += 1
    print(f"{autoscaler}: {count}")


globeco-confirmation-service-74dd5df469-27rg6 trial0004 1 keda
globeco-confirmation-service-74dd5df469-69qpx trial0004 1 keda
globeco-confirmation-service-74dd5df469-8rhmf trial0004 1 keda
globeco-confirmation-service-74dd5df469-9xcsr trial0004 1 keda
globeco-confirmation-service-74dd5df469-h7f6v trial0004 1 keda
globeco-confirmation-service-74dd5df469-j5blb trial0004 1 keda
globeco-confirmation-service-74dd5df469-j5blb trial0004 1 keda
globeco-confirmation-service-74dd5df469-lkxfp trial0004 1 keda
globeco-confirmation-service-74dd5df469-lqr2x trial0004 1 keda
globeco-confirmation-service-74dd5df469-qzzsv trial0004 1 keda
globeco-confirmation-service-74dd5df469-qzzsv trial0004 1 keda
globeco-confirmation-service-74dd5df469-xc8bp trial0004 1 keda
globeco-confirmation-service-74dd5df469-xf6wn trial0004 1 keda
globeco-confirmation-service-74dd5df469-z45t4 trial0004 1 keda
globeco-execution-service-6df778fb86-smsdl trial0004 2 keda
globeco-fix-engine-d95677f59-2fhk2 trial0004 1 keda
globec

In [172]:
df = pd.DataFrame(results)
# display all the rows of the dataframe
with pd.option_context('display.max_rows', None):
    display(df[df.trial_id == "trial0004"])

,action,type,reason,count,first_timestamp,last_timestamp,trial_start_time,trial_end_time,duration,kind,name,namespace,message,trial_id,autoscaler
25,None,Warning,FailedPreStopHook,1,2026-09-18 23:41:30+00:00,2026-09-18 23:41:30+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days 00:00:00,Pod,globeco-confirmation-service-74dd5df469-27rg6,globeco,PreStopHook failed,trial0004,keda
26,None,Warning,FailedPreStopHook,1,2026-09-18 23:39:45+00:00,2026-09-18 23:39:45+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days 00:00:00,Pod,globeco-confirmation-service-74dd5df469-69qpx,globeco,PreStopHook failed,trial0004,keda
27,None,Warning,FailedPreStopHook,1,2026-09-18 23:32:45+00:00,2026-09-18 23:32:45+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days 00:00:00,Pod,globeco-confirmation-service-74dd5df469-8rhmf,globeco,PreStopHook failed,trial0004,keda
28,None,Warning,FailedPreStopHook,1,2026-09-18 23:38:00+00:00,2026-09-18 23:38:00+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days 00:00:00,Pod,globeco-confirmation-service-74dd5df469-9xcsr,globeco,PreStopHook failed,trial0004,keda
29,None,Warning,FailedPreStopHook,1,2026-09-18 23:32:45+00:00,2026-09-18 23:32:45+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days 00:00:00,Pod,globeco-confirmation-service-74dd5df469-h7f6v,globeco,PreStopHook failed,trial0004,keda
30,None,Warning,FailedPreStopHook,1,2026-09-18 23:48:31+00:00,2026-09-18 23:48:31+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days 00:00:00,Pod,globeco-confirmation-service-74dd5df469-j5blb,globeco,PreStopHook failed,trial0004,keda
31,None,Warning,Unhealthy,1,2026-09-18 23:48:31+00:00,2026-09-18 23:48:31+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days 00:00:00,Pod,globeco-confirmation-service-74dd5df469-j5blb,globeco,"Readiness probe failed: Get ""http://10.244.2.6...",trial0004,keda
32,None,Warning,FailedPreStopHook,1,2026-09-18 23:32:45+00:00,2026-09-18 23:32:45+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days 00:00:00,Pod,globeco-confirmation-service-74dd5df469-lkxfp,globeco,PreStopHook failed,trial0004,keda
33,None,Warning,FailedPreStopHook,1,2026-09-18 23:38:00+00:00,2026-09-18 23:38:00+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days 00:00:00,Pod,globeco-confirmation-service-74dd5df469-lqr2x,globeco,PreStopHook failed,trial0004,keda
34,None,Warning,FailedPreStopHook,1,2026-09-18 23:30:15+00:00,2026-09-18 23:30:15+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days 00:00:00,Pod,globeco-confirmation-service-74dd5df469-qzzsv,globeco,PreStopHook failed,trial0004,keda


In [173]:
with pd.option_context('display.max_rows', None):
    display(df[df.reason == "FailedPreStopHook"])

,action,type,reason,count,first_timestamp,last_timestamp,trial_start_time,trial_end_time,duration,kind,name,namespace,message,trial_id,autoscaler
25,None,Warning,FailedPreStopHook,1,2026-09-18 23:41:30+00:00,2026-09-18 23:41:30+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days,Pod,globeco-confirmation-service-74dd5df469-27rg6,globeco,PreStopHook failed,trial0004,keda
26,None,Warning,FailedPreStopHook,1,2026-09-18 23:39:45+00:00,2026-09-18 23:39:45+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days,Pod,globeco-confirmation-service-74dd5df469-69qpx,globeco,PreStopHook failed,trial0004,keda
27,None,Warning,FailedPreStopHook,1,2026-09-18 23:32:45+00:00,2026-09-18 23:32:45+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days,Pod,globeco-confirmation-service-74dd5df469-8rhmf,globeco,PreStopHook failed,trial0004,keda
28,None,Warning,FailedPreStopHook,1,2026-09-18 23:38:00+00:00,2026-09-18 23:38:00+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days,Pod,globeco-confirmation-service-74dd5df469-9xcsr,globeco,PreStopHook failed,trial0004,keda
29,None,Warning,FailedPreStopHook,1,2026-09-18 23:32:45+00:00,2026-09-18 23:32:45+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days,Pod,globeco-confirmation-service-74dd5df469-h7f6v,globeco,PreStopHook failed,trial0004,keda
30,None,Warning,FailedPreStopHook,1,2026-09-18 23:48:31+00:00,2026-09-18 23:48:31+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days,Pod,globeco-confirmation-service-74dd5df469-j5blb,globeco,PreStopHook failed,trial0004,keda
32,None,Warning,FailedPreStopHook,1,2026-09-18 23:32:45+00:00,2026-09-18 23:32:45+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days,Pod,globeco-confirmation-service-74dd5df469-lkxfp,globeco,PreStopHook failed,trial0004,keda
33,None,Warning,FailedPreStopHook,1,2026-09-18 23:38:00+00:00,2026-09-18 23:38:00+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days,Pod,globeco-confirmation-service-74dd5df469-lqr2x,globeco,PreStopHook failed,trial0004,keda
34,None,Warning,FailedPreStopHook,1,2026-09-18 23:30:15+00:00,2026-09-18 23:30:15+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days,Pod,globeco-confirmation-service-74dd5df469-qzzsv,globeco,PreStopHook failed,trial0004,keda
36,None,Warning,FailedPreStopHook,1,2026-09-18 23:48:16+00:00,2026-09-18 23:48:16+00:00,2026-09-18 23:24:46.327266+00:00,2026-09-18 23:54:50.591000+00:00,0 days,Pod,globeco-confirmation-service-74dd5df469-xc8bp,globeco,PreStopHook failed,trial0004,keda
